###  import and   load  the  cleaned  data sets   

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

df = pd.read_csv("../data/cleaned_data.csv")
print("Shape:", df.shape)
df.head()   

Shape: (999769, 10)


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


###  Feature creation: Rainfall intensity   

In [2]:
# Rainfall per day of growing season — captures intensity rather than just total volume
df["Rainfall_per_Day"] = df["Rainfall_mm"] / df["Days_to_Harvest"]

df[["Rainfall_mm", "Days_to_Harvest", "Rainfall_per_Day"]].head()

,Rainfall_mm,Days_to_Harvest,Rainfall_per_Day
0,897.077239,122,7.353092
1,992.673282,140,7.090523
2,147.998025,106,1.396208
3,986.866331,146,6.759358
4,730.379174,110,6.639811


###    Feature creation: Fertilizer × Irrigation interaction

In [3]:
def combo_label(row):
    if row["Fertilizer_Used"] and row["Irrigation_Used"]:
        return "Both"
    elif row["Fertilizer_Used"]:
        return "Fertilizer_Only"
    elif row["Irrigation_Used"]:
        return "Irrigation_Only"
    else:
        return "Neither"

df["Fertilizer_Irrigation_Combo"] = df.apply(combo_label, axis=1)
df["Fertilizer_Irrigation_Combo"].value_counts()

Fertilizer_Irrigation_Combo
Neither            250431
Both               250093
Fertilizer_Only    249847
Irrigation_Only    249398
Name: count, dtype: int64

### Feature creation: Temperature deviation from crop-specific optimum   

In [4]:
# Approximate optimal growing temperatures per crop (domain knowledge, in Celsius)
optimal_temp = {
    "Wheat": 22, "Rice": 27, "Maize": 25,
    "Barley": 20, "Cotton": 28, "Soybean": 26
}

df["Optimal_Temp"] = df["Crop"].map(optimal_temp)
df["Temperature_Deviation"] = (df["Temperature_Celsius"] - df["Optimal_Temp"]).abs()

df[["Crop", "Temperature_Celsius", "Optimal_Temp", "Temperature_Deviation"]].head(10)

,Crop,Temperature_Celsius,Optimal_Temp,Temperature_Deviation
0,Cotton,27.676966,28,0.323034
1,Rice,18.026142,27,8.973858
2,Barley,29.794042,20,9.794042
3,Soybean,16.644190,26,9.355810
4,Wheat,31.620687,22,9.620687
5,Soybean,37.704974,26,11.704974
6,Wheat,31.593431,22,9.593431
7,Rice,30.887107,27,3.887107
8,Wheat,26.752729,22,4.752729
9,Wheat,17.646199,22,4.353801


### Feature creation: Growing season length category    

In [5]:
def season_length_category(days):
    if days < 90:
        return "Short"
    elif days <= 120:
        return "Medium"
    else:
        return "Long"

df["Season_Length_Category"] = df["Days_to_Harvest"].apply(season_length_category)
df["Season_Length_Category"].value_counts()

Season_Length_Category
Medium    345431
Short     332697
Long      321641
Name: count, dtype: int64

In [7]:
###  drop   the  non predictive   columns   
df = df.drop(columns=["Optimal_Temp"])
df.head()  

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare,Rainfall_per_Day,Fertilizer_Irrigation_Combo,Temperature_Deviation,Season_Length_Category
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816,7.353092,Irrigation_Only,0.323034,Long
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341,7.090523,Both,8.973858,Long
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443,1.396208,Neither,9.794042,Medium
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573,6.759358,Irrigation_Only,9.355810,Long
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251,6.639811,Both,9.620687,Medium


###   Encode categorical variables (One-Hot Encoding)

In [8]:
categorical_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition",
                     "Fertilizer_Irrigation_Combo", "Season_Length_Category"]

encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
encoded_array = encoder.fit_transform(df[categorical_cols])
encoded_cols = encoder.get_feature_names_out(categorical_cols)

encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)
df_encoded = pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)

print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

Shape after encoding: (999769, 28)


,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare,Rainfall_per_Day,Temperature_Deviation,Region_North,Region_South,...,Crop_Rice,Crop_Soybean,Crop_Wheat,Weather_Condition_Rainy,Weather_Condition_Sunny,Fertilizer_Irrigation_Combo_Fertilizer_Only,Fertilizer_Irrigation_Combo_Irrigation_Only,Fertilizer_Irrigation_Combo_Neither,Season_Length_Category_Medium,Season_Length_Category_Short
0,897.077239,27.676966,False,True,122,6.555816,7.353092,0.323034,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,992.673282,18.026142,True,True,140,8.527341,7.090523,8.973858,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,147.998025,29.794042,False,False,106,1.127443,1.396208,9.794042,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
3,986.866331,16.644190,False,True,146,6.517573,6.759358,9.355810,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,730.379174,31.620687,True,True,110,7.248251,6.639811,9.620687,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


###   Encode boolean columns as integers

In [9]:
df_encoded["Fertilizer_Used"] = df_encoded["Fertilizer_Used"].astype(int)
df_encoded["Irrigation_Used"] = df_encoded["Irrigation_Used"].astype(int)

df_encoded[["Fertilizer_Used", "Irrigation_Used"]].head()

,Fertilizer_Used,Irrigation_Used
0,0,1
1,1,1
2,0,0
3,0,1
4,1,1


### Separate features and target

In [10]:
target_col = "Yield_tons_per_hectare"

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.columns.tolist()

X shape: (999769, 27)
y shape: (999769,)


['Rainfall_mm',
 'Temperature_Celsius',
 'Fertilizer_Used',
 'Irrigation_Used',
 'Days_to_Harvest',
 'Rainfall_per_Day',
 'Temperature_Deviation',
 'Region_North',
 'Region_South',
 'Region_West',
 'Soil_Type_Clay',
 'Soil_Type_Loam',
 'Soil_Type_Peaty',
 'Soil_Type_Sandy',
 'Soil_Type_Silt',
 'Crop_Cotton',
 'Crop_Maize',
 'Crop_Rice',
 'Crop_Soybean',
 'Crop_Wheat',
 'Weather_Condition_Rainy',
 'Weather_Condition_Sunny',
 'Fertilizer_Irrigation_Combo_Fertilizer_Only',
 'Fertilizer_Irrigation_Combo_Irrigation_Only',
 'Fertilizer_Irrigation_Combo_Neither',
 'Season_Length_Category_Medium',
 'Season_Length_Category_Short']

### Train/test split (before scaling, to avoid data leakage)    

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (799815, 27)
Test shape: (199954, 27)


### Scale numeric features   

In [12]:
numeric_cols_to_scale = ["Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest",
                          "Rainfall_per_Day", "Temperature_Deviation"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test_scaled[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])

X_train_scaled.head()

,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Rainfall_per_Day,Temperature_Deviation,Region_North,Region_South,Region_West,...,Crop_Rice,Crop_Soybean,Crop_Wheat,Weather_Condition_Rainy,Weather_Condition_Sunny,Fertilizer_Irrigation_Combo_Fertilizer_Only,Fertilizer_Irrigation_Combo_Irrigation_Only,Fertilizer_Irrigation_Combo_Neither,Season_Length_Category_Medium,Season_Length_Category_Short
816226,1.097792,1.721608,1,0,-1.059909,1.656524,1.553366,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
180530,1.222346,1.055298,1,0,1.020733,0.314829,1.815579,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
765928,-1.079415,-1.067394,1,1,-1.445213,-0.510638,0.288762,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
207782,1.437720,-0.842185,0,0,0.905141,0.502970,-1.392110,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
411529,-0.751624,-1.263582,1,1,-0.327831,-0.614959,0.160118,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Feature selection check: correlation of new engineered features with target 

In [13]:
check_corr = pd.concat([X_train_scaled[numeric_cols_to_scale], y_train], axis=1)
check_corr.corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)

Rainfall_mm              0.764229
Rainfall_per_Day         0.647108
Temperature_Celsius      0.084577
Temperature_Deviation    0.042285
Days_to_Harvest         -0.002805
Name: Yield_tons_per_hectare, dtype: float64

### Save train/test splits and preprocessing objects    

In [ ]:
import os
os.makedirs("../models/preprocessing", exist_ok=True)

# Save processed datasets
X_train_scaled.to_csv("../data/X_train.csv", index=False)
X_test_scaled.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

# Save preprocessing objects for use in the FastAPI backend later
joblib.dump(encoder, "../models/preprocessing/onehot_encoder.pkl")
joblib.dump(scaler, "../models/preprocessing/scaler.pkl")

# Save the exact column order the model expects (critical for API consistency)
joblib.dump(X_train_scaled.columns.tolist(), "../models/preprocessing/feature_columns.pkl")

print("All preprocessing artifacts saved successfully.")

### Save a feature engineering summary   

In [14]:
import json

fe_summary = {
    "engineered_features": [
        "Rainfall_per_Day",
        "Fertilizer_Irrigation_Combo",
        "Temperature_Deviation",
        "Season_Length_Category"
    ],
    "encoding_method": "One-Hot Encoding (drop first category)",
    "scaling_method": "StandardScaler",
    "final_feature_count": X_train_scaled.shape[1],
    "train_rows": X_train_scaled.shape[0],
    "test_rows": X_test_scaled.shape[0],
    "notes": (
        "Created domain-informed interaction and derived features to capture "
        "relationships not present in raw columns. Scaling fit only on training "
        "data to avoid leakage. Preprocessing objects saved for reuse in the "
        "production API to ensure inference matches training exactly."
    )
}

with open("../reports/feature_engineering_summary.json", "w") as f:
    json.dump(fe_summary, f, indent=4)

fe_summary

{'engineered_features': ['Rainfall_per_Day',
  'Fertilizer_Irrigation_Combo',
  'Temperature_Deviation',
  'Season_Length_Category'],
 'encoding_method': 'One-Hot Encoding (drop first category)',
 'scaling_method': 'StandardScaler',
 'final_feature_count': 27,
 'train_rows': 799815,
 'test_rows': 199954,
 'notes': 'Created domain-informed interaction and derived features to capture relationships not present in raw columns. Scaling fit only on training data to avoid leakage. Preprocessing objects saved for reuse in the production API to ensure inference matches training exactly.'}